# Task 4.1b / 4.1c. GCN trên Graph A và Graph B

| | |
|---|---|
| **Mục đích** | 4.1b: tinh chỉnh GCN-A bằng validation; 4.1c: GCN-B với L1 đã cố định |
| **Đầu vào** | Kaggle Dataset `DiverseVul 9077 - Processed Graph Data` (4.1c: version 1) |
| **Đầu ra** | `results_4_1b_gcn_tuning/<ID>/`: `results.csv`, `per_class.csv`, `confusion_matrices.json`, `training_history.json`, `summary.md`, metadata và trọng số; lựa chọn đã cố định và kết quả test riêng. 4.1c: `results_4_1c_gcn_graph_b/L1/` và kế hoạch riêng |
| **Máy chạy** | Kaggle GPU (T4 hoặc P100), bật Internet |

Hai protocol `project_wise`, `seen_project`; seed 42, 43, 44. Dùng chung model, trainer, metric. Không sửa dữ liệu, vocabulary, nhãn hoặc split. Số cấu hình phải được nhóm thống nhất trước khi chạy. Test chỉ dùng sau khi cố định cấu hình.


In [1]:
from pathlib import Path
import csv
import json
import os
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/meo225/software-bug-detection-using-graphs.git"
RUN_GRAPH_B = Path("/kaggle/input").is_dir()  # True trên Kaggle; False khi kiểm tra offline; đặt False để dùng Graph A
APPROVED_BRANCH = "feature/4_1b_gcn_tuning"
APPROVED_COMMIT = "5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5" if RUN_GRAPH_B else ""  # shared trainer SHA
DATASET_ID = "vtnmai/diversevul-9077-processed-graph-data/versions/1" if RUN_GRAPH_B else ""
APPROVED_CONFIG_IDS = []  # Danh sách đã thống nhất: B0, E1, D1, W1, H1, L1, N1
RUN_TUNING = False
RUN_FINAL_TEST = False
# Để phục hồi hoặc chỉ đánh giá test: trỏ tới output đã lưu, có frozen_selection.json.
SAVED_RESULTS_ROOT = None  # ví dụ /kaggle/input/<saved-output>/results_4_1b_gcn_tuning
OUT_DIR = Path("/kaggle/working/results_4_1b_gcn_tuning")
FINAL_OUT_DIR = Path("/kaggle/working/results_4_1b_gcn_final")
GRAPH_B_ROOT = Path("/kaggle/working/results_4_1c_gcn_graph_b")
GRAPH_B_OUT_DIR = GRAPH_B_ROOT / "L1"
if RUN_GRAPH_B and (RUN_TUNING or RUN_FINAL_TEST or SAVED_RESULTS_ROOT is not None):
    raise ValueError("Graph B phải chạy riêng, không dùng tuning/final/checkpoint Graph A.")
ACTIVE = RUN_TUNING or RUN_FINAL_TEST or RUN_GRAPH_B
if ACTIVE:
    if not Path("/kaggle/input").is_dir():
        raise RuntimeError("Chỉ chạy trên Kaggle với dataset chính thức.")
    if not re.fullmatch(r"[0-9a-f]{40}", APPROVED_COMMIT):
        raise ValueError("Cần SHA đầy đủ đã được phê duyệt.")
    if not re.fullmatch(r"vtnmai/diversevul-9077-processed-graph-data/versions/[1-9][0-9]*", DATASET_ID):
        raise ValueError("Ghi phiên bản Kaggle dataset đã gắn vào notebook.")
REPO_DIR = Path("/tmp/g41c" if RUN_GRAPH_B else "/tmp/g41b") / APPROVED_COMMIT
READ_ROOT = Path(SAVED_RESULTS_ROOT) if SAVED_RESULTS_ROOT else OUT_DIR


## 1. Tải mã nguồn và ghi phiên bản

Nhánh phải có commit đã duyệt trên repository chính thức. Không yêu cầu đã merge vào `main`. Không dùng `git pull`, không cài thư viện trong notebook. Giữ output notebook để lưu nhật ký chạy.

**Task 4.1c — upload trực tiếp:** `RUN_GRAPH_B` tự bật `True` trên Kaggle (có `/kaggle/input`), tự tắt khi kiểm tra offline; giữ `RUN_TUNING=False`, `RUN_FINAL_TEST=False`, `SAVED_RESULTS_ROOT=None`. Notebook này là bản sửa cục bộ chưa commit; không thuộc SHA mã train. Clone nhánh đã publish `feature/4_1b_gcn_tuning`, checkout chính xác shared trainer SHA `5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5`. Không cần publish nhánh 4.1c hoặc chờ merge main. Tham chiếu L1 lấy bằng `git show` từ delivery commit đã publish `e2ccb58def76ca3f3fd329b17627cbff924dd914`, không checkout delivery và không sửa file lịch sử.

Kaggle: bật **GPU**, bật **Internet**, gắn đúng dataset `vtnmai/diversevul-9077-processed-graph-data`, chọn **version 1**. Notebook kiểm tra GPU thực sự hoạt động; clone cần Internet. Version trong cấu hình là khai báo provenance, không tự xác nhận lựa chọn phiên bản trên giao diện Kaggle. Repo tạm nằm trong `/tmp/g41c/<SHA>`, ngoài `/kaggle/working`. Chạy đến cell kế hoạch Graph B, kiểm tra lệnh rồi chạy cell cuối. Lưu executed notebook và Saved Version Kaggle cùng output để giữ provenance notebook upload.

Để dùng lại Graph A: đặt `RUN_GRAPH_B=False`, điền `APPROVED_COMMIT`, `DATASET_ID` và cờ/cấu hình Graph A như trước (final 4.1b dùng SHA thí nghiệm gốc).


In [2]:
if ACTIVE:
    if not REPO_DIR.exists():
        REPO_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "clone", "--no-checkout", "--single-branch", "--branch",
                        APPROVED_BRANCH, REPO_URL, str(REPO_DIR)], check=True, timeout=180)
    subprocess.run(["git", "checkout", "--detach", APPROVED_COMMIT], cwd=REPO_DIR, check=True)
    actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
    assert actual == APPROVED_COMMIT
    if RUN_GRAPH_B:
        assert actual == "5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5", "Sai shared trainer SHA."
    sys.path.insert(0, str(REPO_DIR))
    import torch
    from src.training.gcn_tuning import (
        BASELINE, CANDIDATES, HYPOTHESES, PROTOCOLS, SEEDS,
        configuration, code_identity, environment, freeze_winner, write_json,
        reuse_candidate, read_frozen, validate_context, validate_final,
        digest, data_identity, validate_artifacts,
    )
    from src.graph.processed import ids_hash, load_schema
    assert torch.cuda.is_available(), "Bật GPU trong Kaggle Settings."
    assert torch.ones(1, device="cuda").sum().item() == 1, "Không khởi tạo được GPU."
    assert not code_identity(REPO_DIR)["dirty"], "Checkout phải sạch."
    print("Revision:", actual, "Environment:", environment(), flush=True)


def run_script(arguments, log_path):
    cmd = [sys.executable, "scripts/train_gnn.py", *arguments]
    print("Command:", cmd, flush=True)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("x", encoding="utf-8") as handle:
        proc = subprocess.Popen(cmd, cwd=REPO_DIR, text=True, stdout=subprocess.PIPE,
                                stderr=subprocess.STDOUT,
                                env={**os.environ, "PYTHONUNBUFFERED": "1"})
        for line in proc.stdout:
            print(line, end="", flush=True)
            handle.write(line)
            handle.flush()
        if proc.wait():
            raise RuntimeError(f"Script thất bại; giữ output và xem {log_path}")


Cloning into '/tmp/g41c/5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5'...
HEAD is now at 5d8b99e feat(gnn): implement Task 4.1b GCN tuning


Revision: 5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5 Environment: {'python': '3.13.16', 'torch': '2.11.0+cu130', 'numpy': '2.1.3', 'sklearn': '1.6.1', 'cuda': '13.0', 'gpu': 'Tesla T4'}


## 2. Dữ liệu và cấu hình

Mỗi biến thể chỉ đổi một yếu tố so với B0: E1 dừng theo validation loss; D1 dropout 0,4; W1 weight decay 1e-3; H1 dim 64; L1 hai lớp GCN; N1 tối đa 1.000 node. E1 vẫn giữ mô hình có Macro-F1 validation chưa làm mượt cao nhất. Các cấu hình được so bằng trung bình Macro-F1 validation `project_wise`; `seen_project` là đối chứng.

B0 của 4.1a là tham chiếu lịch sử. Chỉ tái sử dụng làm đối chứng trực tiếp sau khi xác minh tính tương đương của dữ liệu, mã và môi trường. Các CSV cũ không có trọng số hoặc đầy đủ provenance để dùng trực tiếp cho bước đánh giá test của notebook này. Mặc định đề xuất chạy B0 trong cùng môi trường; đây không phải ngân sách đã duyệt.


In [3]:
if ACTIVE:
    found = [p.parent for p in Path("/kaggle/input").rglob("metadata.json")
             if all((p.parent / protocol / "vocab.json").is_file() for protocol in PROTOCOLS)]
    assert len(found) == 1, "Gắn đúng một phiên bản dataset processed chính thức."
    PROCESSED = found[0]
    metadata = json.loads((PROCESSED / "metadata.json").read_text(encoding="utf-8"))
    assert metadata.get("spec_version") in {"1.1", "1.2"}
    assert metadata.get("n_valid_samples") == 7799
    assert sorted(load_schema(REPO_DIR / "configs/graph/schema_spec.yaml")["cwe_mapping"].values()) == list(range(23))
    from collections import Counter
    expected = {"project_wise": {"train": 6101, "validation": 762, "test": 936},
                "seen_project": {"train": 6272, "validation": 764, "test": 763}}
    for protocol in PROTOCOLS:
        with (PROCESSED / protocol / "manifest.csv").open(encoding="utf-8", newline="") as handle:
            manifest = list(csv.DictReader(handle))
        counts = Counter(row["split"] for row in manifest)
        assert len({row["sample_id"] for row in manifest}) == len(manifest)
        assert all(0 <= int(row["y"]) < 23 for row in manifest)
        vocab = json.loads((PROCESSED / protocol / "vocab.json").read_text(encoding="utf-8"))
        assert isinstance(vocab["k"], int) and vocab["k"] > 0
        for name in ("node_type", "token"):
            assert sorted(vocab[name].values()) == list(range(len(vocab[name])))
            assert vocab[name].get("<PAD>") == 0 and vocab[name].get("<UNK>") == 1
        assert vocab["token"].get("<MASK>") == 2
        assert vocab.get("train_ids_sha256") == ids_hash(row["sample_id"] for row in manifest if row["split"] == "train")
        assert dict(counts) == expected[protocol]
        for split in expected[protocol]:
            graph = "graph_b" if RUN_GRAPH_B else "graph_a"
            assert (PROCESSED / protocol / f"{graph}_{split}.pt").is_file()
    print("Dataset:", DATASET_ID, PROCESSED)
if RUN_TUNING:
    assert APPROVED_CONFIG_IDS and len(set(APPROVED_CONFIG_IDS)) == len(APPROVED_CONFIG_IDS)
    assert all(key in CANDIDATES for key in APPROVED_CONFIG_IDS)
    for key in APPROVED_CONFIG_IDS:
        cfg = configuration(key)
        changed = {k: v for k, v in cfg.items() if v != BASELINE[k]}
        assert len(changed) == (0 if key == "B0" else 1)
        print(key, changed, HYPOTHESES[key], flush=True)
    print("Số cấu hình:", len(APPROVED_CONFIG_IDS), "Số lần train tối đa:", 6 * len(APPROVED_CONFIG_IDS))


Dataset: vtnmai/diversevul-9077-processed-graph-data/versions/1 /kaggle/input/datasets/vtnmai/diversevul-9077-processed-graph-data/processed


## 3. Train chỉ validation

`RUN_TUNING=False` bỏ qua bước này. Khi bật, chạy cấu hình chưa có output; cấu hình đã hoàn tất chỉ được dùng lại sau khi kiểm tra đủ artifact, metadata, code, dataset và môi trường. Có thể đặt `SAVED_RESULTS_ROOT` tới output của phiên trước: cấu hình hợp lệ được chép sang `OUT_DIR` mới, không train lại. Output thiếu hoặc không nhất quán gây lỗi rõ ràng; giữ nguyên để kiểm tra, chọn thư mục output mới khi cần chạy lại cấu hình dở dang. Không tự tiếp tục optimizer của lần train bị ngắt. Không nạp test ở bước này.


In [4]:
if RUN_TUNING:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    plan = {
        "code": code_identity(REPO_DIR), "dataset_id": DATASET_ID,
        "configurations": {key: configuration(key) for key in APPROVED_CONFIG_IDS},
        "hypotheses": {key: HYPOTHESES[key] for key in APPROVED_CONFIG_IDS},
        "protocols": list(PROTOCOLS), "seeds": list(SEEDS),
        "selection": "project_wise.validation.macro_f1.mean",
        "checkpoint_selection": "maximum raw validation Macro-F1; earliest tie",
    }
    plan_path = OUT_DIR / "experiment_plan.json"
    if plan_path.exists():
        if json.loads(plan_path.read_text(encoding="utf-8")) != plan:
            raise ValueError("Kế hoạch đã lưu khác cấu hình hiện tại; không ghi đè.")
    else:
        with plan_path.open("x", encoding="utf-8") as handle:
            json.dump(plan, handle, ensure_ascii=False, indent=2)
    for key in APPROVED_CONFIG_IDS:
        target = OUT_DIR / key
        source = target if target.exists() else READ_ROOT / key
        if source.exists():
            reuse_candidate(source, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID)
            if source.resolve() != target.resolve():
                shutil.copytree(source, target)  # destination must be unused
        else:
            args = ["--processed-dir", str(PROCESSED), "--out-dir", str(target),
                    "--models", "gcn", "--graphs", "graph_a", "--protocols", *PROTOCOLS,
                    "--seeds", *map(str, SEEDS), "--validation-only", "--save-checkpoints",
                    "--config-id", key, "--dataset-id", DATASET_ID]
            for name, value in configuration(key).items():
                if value is not None:
                    args += ["--" + name.replace("_", "-"), str(value)]
            run_script(args, OUT_DIR / f"{key}.log")
            reuse_candidate(target, key, REPO_DIR, PROCESSED,
                            REPO_DIR / "configs/graph/schema_spec.yaml", DATASET_ID)


## 4. Cố định lựa chọn bằng validation

Xếp hạng bằng trung bình Macro-F1 validation chưa làm tròn của `project_wise` trên ba seed. Khi bằng nhau ưu tiên B0, E1, D1, W1, H1, L1, N1. Dùng cùng cấu hình cho cả hai protocol. Nếu đã có lựa chọn, chỉ nạp và xác minh; không thay bằng cấu hình khác. Không chọn theo test. Nếu không chạy tuning, bước này chỉ nạp lựa chọn khi `RUN_FINAL_TEST=True`.


In [5]:
if RUN_TUNING or RUN_FINAL_TEST:
    schema = REPO_DIR / "configs/graph/schema_spec.yaml"
    FROZEN_ROOT = OUT_DIR if RUN_TUNING else READ_ROOT
    FROZEN = FROZEN_ROOT / "frozen_selection.json"
    saved_frozen = READ_ROOT / "frozen_selection.json"
    if RUN_TUNING and not FROZEN.exists() and saved_frozen.exists():
        previous = read_frozen(saved_frozen, REPO_DIR, PROCESSED, schema, READ_ROOT)
        if {r["config_id"] for r in previous["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không tự chọn lại.")
        with FROZEN.open("xb") as handle:
            handle.write(saved_frozen.read_bytes())
    if FROZEN.exists():
        frozen = read_frozen(FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
        if RUN_TUNING and {r["config_id"] for r in frozen["ranking"]} != set(APPROVED_CONFIG_IDS):
            raise ValueError("Danh sách cấu hình khác lựa chọn đã cố định; không ghi đè.")
    elif RUN_TUNING:
        frozen = freeze_winner([OUT_DIR / key for key in APPROVED_CONFIG_IDS], FROZEN)
    else:
        raise FileNotFoundError(f"Thiếu lựa chọn validation đã cố định: {FROZEN}")
    validate_context(frozen["winner"]["metadata"], REPO_DIR, PROCESSED, schema, DATASET_ID)
    print("Validation ranking:", frozen["ranking"], flush=True)
    print("Frozen configuration:", frozen["winner"]["config_id"], flush=True)


## 5. Đánh giá test của cấu hình đã cố định

`RUN_FINAL_TEST=False` bỏ qua test. Trong phiên riêng: gắn dataset processed và output tuning đã lưu, đặt `SAVED_RESULTS_ROOT`, cùng commit và dataset version, giữ `RUN_TUNING=False`, bật `RUN_FINAL_TEST=True`. Script xác minh lựa chọn và nạp trọng số theo results root mới, không train lại. `FINAL_OUT_DIR` phải là thư mục ghi được, riêng với output tuning. Nếu đã có kết quả final hợp lệ thì xác minh và dùng lại; không ghi đè. Kết quả final dở dang gây lỗi và cần một thư mục final mới. Báo cạnh 4.1a và token baseline, nêu rõ seen_project có 5 validation và 6 test trùng input với train.


In [6]:
if RUN_FINAL_TEST:
    if FINAL_OUT_DIR.exists() and any(FINAL_OUT_DIR.iterdir()):
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
    else:
        run_script(["--processed-dir", str(PROCESSED), "--out-dir", str(FINAL_OUT_DIR),
                    "--frozen-selection", str(FROZEN), "--results-root", str(FROZEN_ROOT)],
                   FINAL_OUT_DIR.parent / (FINAL_OUT_DIR.name + ".log"))
        validate_final(FINAL_OUT_DIR, FROZEN, REPO_DIR, PROCESSED, schema, FROZEN_ROOT)
    print((FINAL_OUT_DIR / "summary.md").read_text(encoding="utf-8"))


## 6. Task 4.1c — GCN-B với L1 đã cố định

Chỉ chạy khi `RUN_GRAPH_B=True`. Không chạy tìm kiếm bảy ứng viên, không chọn lại L1, không nạp trọng số Graph A và không dùng `--frozen-selection`. Đọc cấu hình từ Git object frozen-selection lịch sử có SHA-256 cố định. `code` ghi commit shared trainer 4.1b; `notebook_provenance` ghi riêng notebook upload chưa commit và băm lịch sử cell đã thực thi. Không gán notebook 4.1c cho SHA lịch sử.

Lệnh dùng pipeline chung: 2 protocol × 3 seed = **6 lần train mới và 6 lần đánh giá test**, không giới hạn mẫu/node. Mỗi lần train chọn checkpoint theo Macro-F1 validation chưa làm tròn cao nhất (tie: epoch sớm nhất), sau đó đánh giá test một lần. Pipeline thường nạp tensor test trước train nhưng chỉ đánh giá sau khi chọn checkpoint; không dùng test để chọn epoch/cấu hình. Cạnh và model giữ nguyên cách xử lý của 4.1b.

Output 4.1c phải mới; nếu phiên dở dang, giữ nguyên output để kiểm tra và chọn `GRAPH_B_ROOT` mới. Không tự resume hoặc ghi đè. Checkpoint, CSV, lịch sử, summary và metadata do script chung xuất. Confusion matrix theo quy ước hiện tại: seed 42 cho mỗi protocol/split. Summary do script chung vẫn có tiêu đề Task 4.1a; kế hoạch riêng bên dưới xác định đúng Task 4.1c. So sánh với GCN-A L1, mặc định 4.1a và token baseline; ghi giới hạn seen_project có 5 validation và 6 test trùng input train.


In [7]:
if RUN_GRAPH_B:
    assert DATASET_ID == "vtnmai/diversevul-9077-processed-graph-data/versions/1"
    assert list(PROTOCOLS) == ["project_wise", "seen_project"] and list(SEEDS) == [42, 43, 44]
    import hashlib
    frozen_reference = "e2ccb58def76ca3f3fd329b17627cbff924dd914:reports/graph/4_1b_results/tuning/frozen_selection.json"
    frozen_sha = "47b7ae96a1918813aa37b36416492ce355045a8001a794857bc3a4bf9fa88900"
    frozen_bytes = subprocess.check_output(["git", "show", frozen_reference], cwd=REPO_DIR)
    assert hashlib.sha256(frozen_bytes).hexdigest() == frozen_sha, "Frozen selection 4.1b đã thay đổi."
    reference = json.loads(frozen_bytes)
    winner = reference["winner"]
    previous = winner["metadata"]["arguments"]
    assert reference["format_version"] == 2
    assert reference["selection_metric"] == "project_wise.validation.macro_f1.mean"
    assert winner["config_id"] == previous["config_id"] == "L1"
    assert previous["dataset_id"] == DATASET_ID and previous["limit"] is None
    assert previous["protocols"] == list(PROTOCOLS) and previous["seeds"] == list(SEEDS)
    graph_b_cfg = {name: previous[name] for name in BASELINE}
    assert graph_b_cfg == configuration("L1"), "L1 hiện tại khác cấu hình đã cố định."
    assert graph_b_cfg["max_nodes"] is None
    assert GRAPH_B_ROOT.is_absolute()
    for historical in (OUT_DIR, FINAL_OUT_DIR):
        assert not (GRAPH_B_ROOT.resolve().is_relative_to(historical.resolve())
                    or historical.resolve().is_relative_to(GRAPH_B_ROOT.resolve()))
    graph_b_args = ["--processed-dir", str(PROCESSED), "--out-dir", str(GRAPH_B_OUT_DIR),
                    "--models", "gcn", "--graphs", "graph_b", "--protocols", *PROTOCOLS,
                    "--seeds", *map(str, SEEDS), "--save-checkpoints",
                    "--config-id", "L1", "--dataset-id", DATASET_ID]
    for name, value in graph_b_cfg.items():
        if value is not None:
            graph_b_args += ["--" + name.replace("_", "-"), str(value)]
    graph_b_plan = {
        "task": "4.1c", "code": code_identity(REPO_DIR), "environment": environment(),
        "dataset_id": DATASET_ID, "processed_dir": str(PROCESSED),
        "graph": "graph_b", "model": "gcn", "config_id": "L1", "configuration": graph_b_cfg,
        "protocols": list(PROTOCOLS), "seeds": list(SEEDS), "limit": None,
        "training_runs": 6, "test_evaluations": 6,
        "notebook_provenance": {
            "origin": "direct Kaggle upload of uncommitted Task 4.1c notebook",
            "source_commit": None,
            "execution_inputs_sha256": hashlib.sha256(json.dumps(
                get_ipython().history_manager.input_hist_raw, ensure_ascii=False).encode("utf-8")).hexdigest(),
            "archive": "Save executed notebook and Kaggle Saved Version with outputs",
        },
        "source_branch": APPROVED_BRANCH,
        "frozen_selection_reference": frozen_reference,
        "frozen_selection_sha256": frozen_sha, "selection_metric": reference["selection_metric"],
        "original_selection_score": winner["score"],
        "original_experiment_code": winner["metadata"]["code"],
        "checkpoint_selection": "maximum raw validation Macro-F1; earliest tie",
        "test_policy": "evaluate once per validation-selected model; never select using test",
        "schema_sha256": digest(REPO_DIR / "configs/graph/schema_spec.yaml"),
        "inputs": {p: data_identity(PROCESSED, p, "graph_b") for p in PROTOCOLS},
        "test_sha256": {p: digest(PROCESSED / p / "graph_b_test.pt") for p in PROTOCOLS},
        "output_dir": str(GRAPH_B_OUT_DIR), "arguments": graph_b_args,
    }
    print("Graph B plan:", json.dumps(graph_b_plan, ensure_ascii=False, indent=2))


Graph B plan: {
  "task": "4.1c",
  "code": {
    "commit": "5d8b99e821c0eb93d873f6ff17c3c7fe8ab5d5f5",
    "dirty": false,
    "source_sha256": {
      "src/models/gnn.py": "209900236ceb90a21fa46fe0038af4d60314bf5ac07d79a48476aff11243f0d9",
      "src/training/gnn_trainer.py": "c82156144ae7af36bda0431bf28287457fd83d817ed1605e0290ae2ce519bced",
      "src/training/gcn_tuning.py": "e9edc7001acc79542fa6c4d0d50ce34fd49ef43e59c205287f6941e4c5f67fc0",
      "src/evaluation/metrics.py": "56b295671f2a4d90ef416355aac488019799529531705d8311ad0eefc4c00e65",
      "scripts/train_gnn.py": "654f12fa16193e2adbec90a9b3bdccb8a96b8bb4af12cefc7fb6d18e8f085b64"
    }
  },
  "environment": {
    "python": "3.13.16",
    "torch": "2.11.0+cu130",
    "numpy": "2.1.3",
    "sklearn": "1.6.1",
    "cuda": "13.0",
    "gpu": "Tesla T4"
  },
  "dataset_id": "vtnmai/diversevul-9077-processed-graph-data/versions/1",
  "processed_dir": "/kaggle/input/datasets/vtnmai/diversevul-9077-processed-graph-data/processed",

### 6.1. Thực thi Graph B và kiểm tra artifact

Chạy cell này sau khi kiểm tra kế hoạch/lệnh ở trên. Metadata của script ghi commit và môi trường thực thi; `experiment_plan.json` liên kết cấu hình với lựa chọn L1 lịch sử. Không sửa các artifact hoặc metadata 4.1b. Không chạy lại cell để ghi đè kết quả.


In [8]:
if RUN_GRAPH_B:
    if GRAPH_B_ROOT.exists() and any(GRAPH_B_ROOT.iterdir()):
        raise FileExistsError(f"Giữ nguyên kết quả cũ; chọn GRAPH_B_ROOT mới: {GRAPH_B_ROOT}")
    GRAPH_B_ROOT.mkdir(parents=True, exist_ok=True)
    with (GRAPH_B_ROOT / "experiment_plan.json").open("x", encoding="utf-8") as handle:
        json.dump(graph_b_plan, handle, ensure_ascii=False, indent=2, allow_nan=False)
    run_script(graph_b_args, GRAPH_B_ROOT / "graph_b.log")
    graph_b_meta = validate_artifacts(GRAPH_B_OUT_DIR)
    assert graph_b_meta["code"] == graph_b_plan["code"]
    assert graph_b_meta["environment"] == graph_b_plan["environment"]
    recorded = graph_b_meta["arguments"]
    assert all(recorded[k] == v for k, v in graph_b_cfg.items())
    assert recorded["graphs"] == ["graph_b"] and recorded["models"] == ["gcn"]
    assert recorded["config_id"] == "L1" and recorded["dataset_id"] == DATASET_ID
    assert recorded["protocols"] == list(PROTOCOLS) and recorded["seeds"] == list(SEEDS)
    assert recorded["save_checkpoints"] and not recorded["validation_only"]
    assert recorded["frozen_selection"] is None and recorded["limit"] is None
    with (GRAPH_B_OUT_DIR / "results.csv").open(encoding="utf-8", newline="") as handle:
        graph_b_rows = list(csv.DictReader(handle))
    assert len(graph_b_rows) == 12
    assert {(r["protocol"], int(r["seed"]), r["split"]) for r in graph_b_rows} == {
        (p, s, split) for p in PROTOCOLS for s in SEEDS for split in ("validation", "test")}
    assert all(r["graph"] == "graph_b" and r["model"] == "gcn" for r in graph_b_rows)
    runs = graph_b_meta["runs"]
    assert len(runs) == 6 and {(r["protocol"], r["seed"]) for r in runs} == {
        (p, s) for p in PROTOCOLS for s in SEEDS}
    for run in runs:
        assert run["checkpoint"] == f"{run['protocol']}_graph_b_gcn_{run['seed']}.pt"
        assert digest(GRAPH_B_OUT_DIR / run["checkpoint"]) == run["sha256"]
    print((GRAPH_B_OUT_DIR / "summary.md").read_text(encoding="utf-8"))


Command: ['/usr/bin/python3', 'scripts/train_gnn.py', '--processed-dir', '/kaggle/input/datasets/vtnmai/diversevul-9077-processed-graph-data/processed', '--out-dir', '/kaggle/working/results_4_1c_gcn_graph_b/L1', '--models', 'gcn', '--graphs', 'graph_b', '--protocols', 'project_wise', 'seen_project', '--seeds', '42', '43', '44', '--save-checkpoints', '--config-id', 'L1', '--dataset-id', 'vtnmai/diversevul-9077-processed-graph-data/versions/1', '--dim', '128', '--num-layers', '2', '--dropout', '0.2', '--lr', '0.001', '--weight-decay', '0.0001', '--batch-size', '32', '--max-epochs', '40', '--patience', '8', '--early-stopping', 'macro_f1']
Thiết bị: cuda; torch 2.11.0+cu130
project_wise/graph_b: train=6101, validation=762, test=936; K=23
  [gcn seed=42] epoch 01 loss=3.1341 val_macro_f1=0.0254
  [gcn seed=42] epoch 02 loss=2.9965 val_macro_f1=0.0065
  [gcn seed=42] epoch 03 loss=2.7420 val_macro_f1=0.0282
  [gcn seed=42] epoch 04 loss=2.5031 val_macro_f1=0.0284
  [gcn seed=42] epoch 05 lo